# 07 · Model export (ONNX + metadata for the website)

Exports each Random Forest to ONNX for `onnxruntime-web`, and writes the
metadata the browser needs to reproduce training-time features exactly.

Key point: the exported pipeline **includes** the median imputer, so the browser
must NOT impute - it sends a complete float32 feature vector, or NaN, and the
graph handles it.

## Shared setup (run first)

This notebook uses the modules in `ml/`. Upload them once to the Colab runtime:

1. Click the folder icon in the left sidebar (Files).
2. Click **Upload to session storage** and select `extract_pose.py`,
   `fitness_features.py`, `build_dataset.py`, `train_baseline.py`,
   `train_temporal.py`, `export_onnx.py` from `F:\Fitness Coach\ml`.
3. Run the cell below.

The dataset zip is expected at
`/content/drive/MyDrive/Dataset Exercise Quality-wise.zip` (Google Drive).

In [ ]:
import os, sys, subprocess, json, glob
from pathlib import Path

ROOT = Path('/content/fitness')
ROOT.mkdir(parents=True, exist_ok=True)

# The uploaded modules land in /content; mirror them into the work root.
for f in ['extract_pose.py', 'fitness_features.py', 'build_dataset.py', 'train_baseline.py',
          'experiment_generalization.py', 'experiment_clip.py', 'experiment_repmetry.py',
          'train_final.py', 'train_temporal.py', 'export_onnx.py']:
    src = Path('/content') / f
    if src.exists():
        dst = ROOT / f
        if not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime:
            dst.write_bytes(src.read_bytes())

sys.path.insert(0, str(ROOT))
print('work root :', ROOT)
print('modules   :', sorted(p.name for p in ROOT.glob('*.py')))
print('cuda      :', subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True).stdout.strip() or 'no GPU')

## 1. Export and verify parity

The final models are produced by `train_final.py`, which fits, thresholds, exports
ONNX and checks sklearn-vs-onnxruntime agreement (including a NaN row) in one go.
Re-run it here if you changed anything upstream.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'skl2onnx', 'onnxruntime'],
                     capture_output=True, text=True).returncode)

In [ ]:
!cd /content/fitness && python -u /content/fitness/train_final.py

## 2. Inspect the exported contract

In [ ]:
import json
meta = json.load(open(ROOT / 'export/model_metadata.json'))
fm = json.load(open(ROOT / 'export/final_metrics.json'))
print('version:', meta['version'], '| unit:', meta['unit'])
print('feature_order length:', len(meta['feature_order']))
for ex, r in fm['exercises'].items():
    print(f"  {ex:11s} {r['onnx_file']:32s} thr={r['threshold']} "
          f"test_auc={r['test']['roc_auc']} parity={r['onnx_parity_max_abs_diff']:.2e}")
print()
print('first 12 features (must match the browser extractor exactly):')
print(meta['feature_order'][:12])

## 3. Copy artefacts to Drive

Colab runtimes are temporary. Copy the shipped models, their metadata and the
split definition to Drive so the website build always has the current version.

In [ ]:
import shutil, os
DEST = '/content/drive/MyDrive/fitness_coach_models'
os.makedirs(DEST + '/export', exist_ok=True)
for f in os.listdir(ROOT / 'export'):
    shutil.copy2(ROOT / 'export' / f, DEST + '/export/' + f)
for f in ['splits.json', 'summary.json', 'clip_cv_results.csv',
          'repmetry_cv_results.csv', 'univariate_auc.csv']:
    p = ROOT / 'dataset' / f
    if p.exists():
        shutil.copy2(p, DEST + '/dataset_' + f)
print('copied to', DEST)
for r, _d, fs in os.walk(DEST):
    for f in fs:
        print('  ', os.path.join(r, f).replace(DEST + '/', ''), round(os.path.getsize(os.path.join(r, f))/1e3, 1), 'kB')

## 4. Wiring into the website

1. Copy `form_classifier_<exercise>.onnx` and `model_metadata.json` into
   `F:\Fitness Coach\public\models\`.
2. In the browser: MediaPipe Pose Landmarker (JS, same `full` model) ->
   `computeFrameFeatures()` ported 1:1 from `ml/fitness_features.py` ->
   rolling 2 s buffer -> the 8 statistics per feature in the same order as
   `meta['feature_order']` -> `ort.InferenceSession`.
3. Compare `probabilities[1]` (GOOD) against `threshold` from the metadata,
   never against 0.5.
4. Rep counting stays a separate exercise-specific state machine in TypeScript;
   do not derive it from the classifier output.
5. Feedback stays rule-based: "hip drops below knee" etc. is observable, while
   "your glutes are weak" is not diagnosable from a phone camera.